# 🧠 Modelo 5: Transfer Learning Avanzado con ResNet50

En este notebook utilizamos la arquitectura **ResNet50** (una de las más potentes en visión por computador) preentrenada en ImageNet. Aplicamos **Learning Rate Scheduling** y evaluamos su desempeño con métricas especializadas (AUC).

---

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from sklearn.metrics import classification_report, confusion_matrix

# Añadir el directorio raíz al path para importar módulos locales
sys.path.append('..')
import oct_dataloader as dataloaders
import modelos.modelo_resnet50 as resnet_model

print("✅ Librerías e importaciones listas")

c:\Users\pablo\miniconda3\envs\oct_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Librerías e importaciones listas


In [2]:
# Configurar GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU detectada(s): {(gpu)}")
    except RuntimeError as e:
        print(e)
else:
    print(" No se detectó GPU. Se usará la CPU.")

GPU detectada(s): PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


## 1. Carga de Datos (RGB para ResNet50)

**IMPORTANTE**: ResNet50 está diseñado para imágenes en color (3 canales). Por ello, configuramos `color_mode='rgb'` en el dataloader.

In [3]:
# Descargar/Localizar dataset
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")
data_path = path
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        data_path = root
        break

# Configuración del DataLoader
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SUBSET = 0.8 # Usamos el 30% del entrenamiento para que sea fluido

train_ds, val_ds, test_ds, class_names = dataloaders.create_oct_dataloaders(
    data_path=data_path,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb', # <--- OBLIGATORIO para ResNet50
    train_subset_fraction=SUBSET, 
    optimize=False
)

⚙️ Configuración de DataLoaders
   • Tamaño de imagen: (224, 224)
   • Batch size: 16
   • Clases: ['CNV', 'DME', 'DRUSEN', 'NORMAL']
   • Train subset: 80.0%
   • Seed: 42

📦 Creando data loader de entrenamiento...
Found 76515 files belonging to 4 classes.
   ⚠️  Usando 80.0% del dataset de entrenamiento
   ⚠️  Batches: 3826 de 4783
✅ Data loader de entrenamiento creado

📦 Creando data loader de validación...
Found 21861 files belonging to 4 classes.
✅ Data loader de validación creado

📦 Creando data loader de prueba...
Found 10933 files belonging to 4 classes.
✅ Data loader de prueba creado

📊 RESUMEN DE DATASETS
Train:      3826 batches
Validation: 1367 batches
Test:       684 batches



## 2. Creación y Compilación del Modelo

ResNet50 tiene millones de parámetros. Congelamos su base para realizar solo **Transfer Learning** sobre las capas superiores que hemos añadido.

In [4]:
from tensorflow.keras.metrics import AUC

# Hiperparámetros configurables
LEARNING_RATE = 0.001
DROPOUT = 0.4

# 1. Crear el modelo con ResNet50
model = resnet_model.create_resnet_model(
    input_shape=(224, 224, 3), # (Ancho, Alto, Canales RGB)
    num_classes=4, 
    dropout_rate=DROPOUT
)

# 2. Compilar especificando métricas (incluyendo el AUC para mayor robustez)
metrics = [
    'accuracy'
]

model = resnet_model.compile_resnet_model(
    model, 
    learning_rate=LEARNING_RATE, 
    metrics=metrics
)

model.summary()

Model: "ResNet50_Transfer"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 batch_normalization (BatchN  (None, 256)              1024      
 ormalization)                                                   
                                                                 
 activation (Activation)     (None, 256)               0         
                                                                 
 dropout (Dropout)           (None, 256)         

## 3. Entrenamiento con Scheduler

Implementamos `ReduceLROnPlateau` para mejorar la estabilidad del entrenamiento.

In [5]:
EPOCHS = 15

# Obtener callbacks avanzados
callbacks = resnet_model.get_resnet_callbacks(
    patience_stop=5, 
    patience_lr=3, 
    factor_lr=0.2
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/15
3826/3826 [==============================] - 282s 72ms/step - loss: 0.4154 - accuracy: 0.8562 - val_loss: 0.3058 - val_accuracy: 0.8943 - lr: 0.0010
Epoch 2/15
3826/3826 [==============================] - 277s 72ms/step - loss: 0.3362 - accuracy: 0.8827 - val_loss: 0.2603 - val_accuracy: 0.9101 - lr: 0.0010
Epoch 3/15
3826/3826 [==============================] - 276s 72ms/step - loss: 0.3104 - accuracy: 0.8926 - val_loss: 0.2686 - val_accuracy: 0.9067 - lr: 0.0010
Epoch 4/15
3826/3826 [==============================] - 268s 70ms/step - loss: 0.2897 - accuracy: 0.8999 - val_loss: 0.2320 - val_accuracy: 0.9210 - lr: 0.0010
Epoch 5/15
3826/3826 [==============================] - 275s 72ms/step - loss: 0.2744 - accuracy: 0.9051 - val_loss: 0.2303 - val_accuracy: 0.9215 - lr: 0.0010
Epoch 6/15
3826/3826 [==============================] - 275s 72ms/step - loss: 0.2616 - accuracy: 0.9080 - val_loss: 0.2268 - val_accuracy: 0.9231 - lr: 0.0010
Epoch 7/15
3826/3826 [==================

## 4. Curvas de Aprendizaje y Evaluación

In [1]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()



plt.subplot(1, 3, 3)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()

plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Resultados finales en el conjunto de Test
print("📊 Evaluación Final (Test Set):")
results = model.evaluate(test_ds)
for i, metric in enumerate(model.metrics_names):
    print(f"{metric.capitalize()}: {results[i]:.4f}")